In [ ]:
import os
import gzip
import numpy as np
import cv2

# =========================
# 1. MOUNT GOOGLE DRIVE
# =========================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except:
    print("⚠️ bukan di colab, skip mount")

# =========================
# 2. PATH DATASET
# =========================
image_dir = "/content/drive/MyDrive/buat preprocessing/dataset 2/stare-images"
label_dir = "/content/drive/MyDrive/buat preprocessing/dataset 2/labels-ah"

out_image_dir = "/content/drive/MyDrive/buat preprocessing/dataset 2/stare-images-224x224"
out_label_dir = "/content/drive/MyDrive/buat preprocessing/dataset 2/labels-ah-224x224"

# =========================
# 3. CEK PATH
# =========================
if not os.path.exists(image_dir):
    raise Exception(f"❌ image_dir ga ketemu: {image_dir}")
if not os.path.exists(label_dir):
    raise Exception(f"❌ label_dir ga ketemu: {label_dir}")

print("✅ path aman")

# =========================
# 4. BUAT FOLDER OUTPUT
# =========================
os.makedirs(out_image_dir, exist_ok=True)
os.makedirs(out_label_dir, exist_ok=True)

# =========================
# 5. LOAD FUNCTION
# =========================
def load_ppm_gz(path):
    with gzip.open(path, 'rb') as f:
        file_bytes = np.frombuffer(f.read(), np.uint8)
        img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    return img

# =========================
# 6. PREPROCESS + SAVE
# =========================
count = 0

for img_name in os.listdir(image_dir):
    if img_name.endswith(".ppm.gz"):
        base = img_name.replace(".ppm.gz", "")

        img_path = os.path.join(image_dir, img_name)
        mask_path = os.path.join(label_dir, base + ".ah.ppm.gz")

        if not os.path.exists(mask_path):
            print(f"⚠️ skip (no label): {base}")
            continue

        # load
        image = load_ppm_gz(img_path)
        mask = load_ppm_gz(mask_path)

        if image is None or mask is None:
            print(f"⚠️ gagal load: {base}")
            continue

        # resize
        image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)

        # mask grayscale
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        # save ke PNG
        cv2.imwrite(os.path.join(out_image_dir, base + ".png"), image)
        cv2.imwrite(os.path.join(out_label_dir, base + ".png"), mask)

        count += 1

print(f"🔥 selesai, total saved: {count}")

Mounted at /content/drive
✅ path aman
🔥 selesai, total saved: 20
